## 1.5 Observaciones iniciales

**Lo que ya podemos ver a simple vista:**

1. **Demanda con tendencia creciente** — Ecuador crece ~3-5% anual en consumo eléctrico
2. **Estacionalidad clara** — Meses de inicio de año tienden a tener menor demanda
3. **Crisis 2024 visible** — Caída de demanda en oct-dic 2024 por racionamientos
4. **Relación inversa hidro/fósil** — Cuando baja hidro, sube fósil (compensación)
5. **CO₂ como proxy** — La intensidad de carbono es un indicador directo de cuándo se usa más fósil
6. **Datos completos** — 85 meses sin interrupciones, alta calidad

**Pregunta clave para el proyecto:** ¿Podemos detectar automáticamente los meses de crisis solo mirando los patrones en los datos, sin saber de antemano cuándo ocurrieron?

---
*Siguiente notebook: 02 — Limpieza y calidad de datos*

In [ ]:
# CO2 intensity — refleja cuánto fósil se está usando
fig = px.line(df, x='fecha', y='co2_intensity_gco2_kwh',
              title='Intensidad de Carbono del Sector Eléctrico (gCO₂/kWh)',
              labels={'co2_intensity_gco2_kwh': 'gCO₂/kWh', 'fecha': ''})
fig.add_vrect(x0='2024-09-15', x1='2024-12-31', fillcolor='rgba(255,0,0,0.15)',
              annotation_text='Crisis: más fósil → más CO₂')
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
# Mix energético: Hidro vs Fósil a lo largo del tiempo
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['fecha'], y=df['gen_hydro'], name='Hidroeléctrica',
                         fill='tozeroy', line=dict(color='#2196F3')))
fig.add_trace(go.Scatter(x=df['fecha'], y=df['gen_fossil'], name='Fósil',
                         fill='tozeroy', line=dict(color='#F44336')))
fig.add_trace(go.Scatter(x=df['fecha'], y=df['gen_wind'], name='Eólica',
                         fill='tozeroy', line=dict(color='#4CAF50')))
fig.update_layout(title='Mix Energético de Ecuador — Generación por Fuente (%)',
                  yaxis_title='% de generación', template='plotly_white',
                  hovermode='x unified')
fig.show()

In [ ]:
# Demanda eléctrica mensual
fig = px.line(df, x='fecha', y='demanda_twh', 
              title='Demanda Eléctrica Mensual de Ecuador (TWh)',
              labels={'demanda_twh': 'Demanda (TWh)', 'fecha': ''})
fig.add_vrect(x0='2024-04-01', x1='2024-06-30', fillcolor='rgba(255,0,0,0.1)', 
              annotation_text='Apagones', annotation_position='top left')
fig.add_vrect(x0='2024-09-15', x1='2024-12-31', fillcolor='rgba(255,0,0,0.2)',
              annotation_text='Crisis oct-dic', annotation_position='top left')
fig.update_layout(template='plotly_white')
fig.show()

## 1.4 Primera visualización: ¿Cómo se ve el sector eléctrico de Ecuador?

Antes de cualquier análisis, grafiquemos las variables clave para tener una intuición visual.

In [ ]:
cols_principales = ['demanda_twh', 'gen_total_generation', 'gen_hydro', 'gen_fossil', 
                    'gen_gas', 'gen_wind', 'gen_solar', 'co2_intensity_gco2_kwh', 
                    'importaciones_netas_twh']
cols_disponibles = [c for c in cols_principales if c in df.columns]

df[cols_disponibles].describe().round(3)

## 1.3 Estadísticas descriptivas

Primer vistazo a los rangos, medias y distribuciones de las variables principales.

### Diccionario de variables

| Variable | Descripción | Unidad | Fuente |
|----------|-------------|--------|--------|
| `gen_hydro` | Generación hidroeléctrica | % del mix | Ember |
| `gen_fossil` | Generación fósil total | % del mix | Ember |
| `gen_gas` | Generación a gas natural | % del mix | Ember |
| `gen_other_fossil` | Petróleo y otros fósiles | % del mix | Ember |
| `gen_bioenergy` | Biomasa/biogás | % del mix | Ember |
| `gen_wind` | Generación eólica | % del mix | Ember |
| `gen_solar` | Generación solar | % del mix | Ember |
| `gen_total_generation` | Generación total | TWh | Ember |
| `demanda_twh` | Demanda eléctrica total | TWh | Ember |
| `co2_intensity_gco2_kwh` | Intensidad de carbono | gCO₂/kWh | Ember |
| `importaciones_netas_twh` | Importaciones netas (Col/Perú) | TWh | Ember |
| `poblacion` | Población estimada | personas | OWID |
| `pib_usd` | PIB nominal | USD | OWID |
| `share_hidro_pct` / `fossil` / `renovable` | Shares anuales históricos | % | OWID |

In [ ]:
# Información de cada columna
info_data = []
for col in df.columns:
    info_data.append({
        'Columna': col,
        'Tipo': str(df[col].dtype),
        'No-Null': df[col].notna().sum(),
        'Nulos': df[col].isna().sum(),
        '% Completo': f"{df[col].notna().mean()*100:.0f}%",
        'Ejemplo': str(df[col].dropna().iloc[0]) if df[col].notna().any() else 'N/A'
    })

pd.DataFrame(info_data)

## 1.2 Estructura del dataset

Veamos qué columnas tenemos, sus tipos y qué significan.

In [ ]:
df = pd.read_csv('../../data/raw/ecuador_electricity_real.csv', parse_dates=['fecha'])

print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Período: {df['fecha'].min().strftime('%B %Y')} → {df['fecha'].max().strftime('%B %Y')}")
print(f"Granularidad: mensual ({df.shape[0]} meses)")
print(f"\nMemoria: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
df.head()

## 1.1 Carga de datos

Cargamos el dataset combinado que construimos desde Ember (mensual) + Our World in Data (anual). El archivo ya integra ambas fuentes.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

# 01 — Carga y Exploración de Datos

## Objetivo
Entender qué datos tenemos disponibles sobre el sector eléctrico ecuatoriano, de dónde vienen, qué estructura tienen y qué podemos hacer con ellos.

## Fuentes de datos
| Fuente | Qué contiene | Granularidad | Cobertura |
|--------|-------------|-------------|-----------|
| **Ember** | Generación por tipo, demanda, emisiones CO₂ | Mensual | 2019-2026 |
| **Our World in Data** | Mix energético, PIB, población | Anual | 1900-2024 |
| **World Bank** | Consumo per cápita, pérdidas | Anual | 2000-2023 |

## Contexto: ¿Por qué Ecuador?
Ecuador depende ~70% de generación **hidroeléctrica** (principalmente la central Coca Codo Sinclair de 1500 MW). Esto lo hace extremadamente sensible a:
- **Sequías** → menos agua → menos generación → apagones
- **Crecimiento de demanda** → infraestructura insuficiente
- **Crisis 2023-2024** → racionamientos de hasta 14 horas diarias